In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def simple_scraper(url):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}

    lista_libros = []
    try:
        contador = 0
        while url:
            book_page = requests.get(url, headers=headers, timeout=10)

            book_page.raise_for_status()

            soup = BeautifulSoup(book_page.text, "lxml")

            libros = soup.find_all("article", class_="product_pod")
            for libro in libros:
                contador += 1
                titulo = libro.find('h3').a['title']
                precio = libro.find("p", class_="price_color").get_text(strip=True)
                rating = libro.find("p", class_="star-rating").get("class")
                disponibilidad = libro.find("p", class_="instock availability").get_text(strip=True)
                link = libro.find("h3").a["href"]
                link_libro = urljoin(url, link)
                lista_libros.append({"Titulo:": titulo, "Precio:": precio, "Rating:": rating[1], "Disponibilidad:":disponibilidad, "Link:":link_libro})
                #print(f"{contador}. Titulo: {titulo} - Precio: {precio} - Rating: {rating[1]} - Disponibilidad: {disponibilidad}")
                print(lista_libros)

            boton_next = soup.find("li", class_="next")
            if boton_next:
                href_next = boton_next.a['href']
                url = urljoin(url, href_next)
            else:
                break

    except requests.exceptions.HTTPError as err:
        print(f"Error HTTP: {err}")
    
    except Exception as e:
        print(f"Ocurrio un error inesperado: {e}")

if __name__ == "__main__":
    URL_OBJETIVO = "https://books.toscrape.com/catalogue/page-1.html"
    simple_scraper(URL_OBJETIVO)